# NEBLI — busca AnKing para *Controle hormonal*

Etapa 1 do deck: encontrar, no AnKing, os candidatos para cada um dos 36 conceitos
do contrato `cobertura-bioq-24-controle-hormonal.json`.

**O que fazer:** Ambiente de execução → Executar tudo. Não precisa editar nada se a
pasta do Drive for a de sempre.

**O que sai:** `candidatos-bioq-24-controle-hormonal.json` na mesma pasta do Drive.
É um arquivo pequeno (só ids, guids, tags e trechos) — me mande que eu faço a
curadoria e devolvo a lista de GUIDs para a extração.

Nada aqui modifica o AnKing. O `.apkg` é aberto como zip e só o membro da coleção
é lido — a mídia (que é o que pesa) nem é tocada nesta etapa.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PASTA = '/content/drive/MyDrive/NEBLI'   # ajuste se a pasta tiver outro nome
APKG  = None                            # detectado automaticamente abaixo
WORK  = '/content/work'
SLUG  = 'bioq-24-controle-hormonal'

if not os.path.isdir(PASTA):
    # procura a pasta que contem o deck mestre
    for raiz, dirs, arqs in os.walk('/content/drive/MyDrive'):
        if any(a.startswith('AnKing Step Deck') and a.endswith('.apkg') for a in arqs):
            PASTA = raiz
            break

cands = [a for a in os.listdir(PASTA) if a.startswith('AnKing Step Deck') and a.endswith('.apkg')]
if not cands:
    raise SystemExit('deck mestre do AnKing nao encontrado em ' + PASTA)
APKG = os.path.join(PASTA, sorted(cands)[-1])

os.makedirs(WORK, exist_ok=True)
print('pasta :', PASTA)
print('mestre:', os.path.basename(APKG), '|', round(os.path.getsize(APKG)/2**30, 2), 'GiB')

In [ ]:
import zipfile, shutil, time

col_path = os.path.join(WORK, 'collection.anki21')

if not os.path.exists(col_path):
    t0 = time.time()
    with zipfile.ZipFile(APKG) as z:
        nomes = set(z.namelist())
        membro = None
        for c in ('collection.anki21', 'collection.anki2'):
            if c in nomes:
                membro = c
                break
        if membro is None:
            achados = [n for n in list(nomes)[:20]]
            raise SystemExit('membro de colecao nao encontrado. amostra: ' + str(achados))
        with z.open(membro) as src, open(col_path, 'wb') as dst:
            shutil.copyfileobj(src, dst, 1024 * 1024)
    print('extraido', membro, 'em', round(time.time() - t0, 1), 's')

print('colecao:', round(os.path.getsize(col_path) / 2**20, 1), 'MiB')

In [ ]:
import sqlite3, json, re, html

con = sqlite3.connect(col_path)
cur = con.cursor()

# nome do notetype: schema antigo (col.models) e novo (tabela notetypes)
modelos = {}
try:
    linha = cur.execute('select models from col').fetchone()
    if linha and linha[0]:
        for mid, m in json.loads(linha[0]).items():
            modelos[int(mid)] = m.get('name', '')
except sqlite3.OperationalError:
    pass
if not modelos:
    for mid, nome in cur.execute('select id, name from notetypes'):
        modelos[mid] = nome

n_cards = dict(cur.execute('select nid, count(*) from cards group by nid'))

TAGS_HTML = re.compile(r'<[^>]+>')
ESPACOS = re.compile(r'\s+')

def limpa(s):
    s = s.replace('\x1f', ' | ')
    s = TAGS_HTML.sub(' ', s)
    s = html.unescape(s)
    return ESPACOS.sub(' ', s).strip()

notas = []
for nid, guid, mid, tags, flds in cur.execute('select id, guid, mid, tags, flds from notes'):
    texto = limpa(flds)
    notas.append({
        'nid': nid,
        'guid': guid,
        'modelo': modelos.get(mid, str(mid)),
        'tags': (tags or '').strip(),
        'texto': texto,
        'lower': texto.lower(),
        'tags_lower': (tags or '').lower(),
        'cards': n_cards.get(nid, 1),
    })

con.close()
print(len(notas), 'notas carregadas')
print('notetypes:', len(modelos))

In [ ]:
# Familias de busca por conceito, extraidas do contrato de cobertura.
# 'q' = termos no corpo do card; 'tag' = pistas na arvore de tags do AnKing.
SPEC = {
 '1.1':  {'nome': 'Tres alavancas de regulacao', 'q': ['allosteric', 'covalent modification', 'phosphorylation', 'dephosphorylation'], 'tag': ['biochem', 'enzyme']},
 '1.2':  {'nome': 'Receptor define a celula-alvo', 'q': ['target cell', 'receptor specificity', 'target tissue'], 'tag': ['endocrine']},
 '1.3':  {'nome': 'Endocrino, paracrino, autocrino', 'q': ['paracrine', 'autocrine', 'endocrine signaling'], 'tag': ['endocrine']},
 '1.4':  {'nome': 'Solubilidade decide o receptor', 'q': ['lipophilic hormone', 'water soluble hormone', 'steroid hormone receptor', 'intracellular receptor'], 'tag': ['endocrine']},
 '1.5':  {'nome': 'Tireoidiano em receptor nuclear', 'q': ['thyroid hormone receptor', 'nuclear receptor thyroid'], 'tag': ['thyroid']},
 '1.6':  {'nome': 'Tipos de transdutor', 'q': ['G protein coupled', 'receptor tyrosine kinase', 'guanylyl cyclase', 'ligand gated'], 'tag': ['signal', 'pharm']},
 '1.7':  {'nome': 'Escala de tempo por receptor', 'q': ['nicotinic', 'muscarinic', 'second messenger seconds'], 'tag': ['pharm', 'receptor']},
 '1.8':  {'nome': 'Amplificacao em cascata', 'q': ['signal amplification', 'second messenger', 'cascade amplif'], 'tag': ['signal']},
 '1.9':  {'nome': 'Alta afinidade do receptor', 'q': ['receptor affinity', 'dissociation constant', 'high affinity receptor'], 'tag': ['pharm']},
 '1.10': {'nome': 'Pontos de regulacao hormonal', 'q': ['hormone synthesis', 'hormone secretion', 'hormone degradation'], 'tag': ['endocrine']},
 '1.11': {'nome': 'Classes quimicas (opcional)', 'q': ['peptide hormone', 'steroid hormone', 'eicosanoid', 'catecholamine'], 'tag': ['endocrine']},
 '2.1':  {'nome': 'Processamento da insulina / peptideo C', 'q': ['proinsulin', 'c-peptide', 'c peptide', 'preproinsulin'], 'tag': ['pancreas', 'endocrine']},
 '2.2':  {'nome': 'Secrecao de insulina na celula beta', 'q': ['glut2', 'glucokinase', 'katp', 'atp sensitive potassium', 'beta cell depolar'], 'tag': ['pancreas']},
 '2.3':  {'nome': 'ATP fecha o canal KATP', 'q': ['closes atp sensitive', 'katp channel clos', 'kir6', 'sur1'], 'tag': ['pancreas']},
 '2.4':  {'nome': 'Glicoquinase vs hexoquinase', 'q': ['glucokinase', 'hexokinase', 'hexokinase iv'], 'tag': ['biochem', 'glycolysis']},
 '2.5':  {'nome': 'Familia GLUT', 'q': ['glut4', 'glut1', 'glut2', 'glut3', 'glucose transporter'], 'tag': ['biochem']},
 '2.6':  {'nome': 'Sulfonilureias', 'q': ['sulfonylurea', 'glyburide', 'glipizide', 'glimepiride'], 'tag': ['pharm', 'diabet']},
 '2.7':  {'nome': 'Diazoxido / insulinoma', 'q': ['diazoxide', 'insulinoma'], 'tag': ['pharm', 'endocrine']},
 '2.8':  {'nome': 'Efeitos da insulina', 'q': ['insulin increases', 'insulin stimulates', 'insulin effect', 'anabolic'], 'tag': ['endocrine', 'biochem']},
 '2.9':  {'nome': 'Glicogenio fosforilase a/b', 'q': ['glycogen phosphorylase', 'phosphorylase kinase', 'phosphorylase a'], 'tag': ['glycogen']},
 '2.10': {'nome': 'Sintase e fosforilase em espelho', 'q': ['glycogen synthase', 'gsk3', 'protein phosphatase 1'], 'tag': ['glycogen']},
 '2.11': {'nome': 'Insulina via PKB/PP1/GSK3', 'q': ['akt', 'protein kinase b', 'phosphodiesterase camp'], 'tag': ['signal', 'biochem']},
 '2.12': {'nome': 'Glucagon: celulas alfa e estimulos', 'q': ['glucagon', 'alpha cell', 'islets of langerhans'], 'tag': ['pancreas']},
 '2.13': {'nome': 'Proglucagon e GLP-1', 'q': ['glp-1', 'glp 1', 'proglucagon', 'incretin'], 'tag': ['pancreas', 'pharm']},
 '2.14': {'nome': 'Cascata do AMPc', 'q': ['adenylate cyclase', 'adenylyl cyclase', 'protein kinase a', 'camp'], 'tag': ['signal']},
 '2.15': {'nome': 'Efeitos do glucagon', 'q': ['glucagon increases', 'gluconeogenesis', 'ketogenesis'], 'tag': ['biochem']},
 '2.16': {'nome': 'Biossintese das catecolaminas', 'q': ['tyrosine hydroxylase', 'dopa decarboxylase', 'dopamine beta hydroxylase', 'pnmt'], 'tag': ['biochem', 'adrenal']},
 '2.17': {'nome': 'Efeitos da epinefrina', 'q': ['epinephrine', 'fight or flight', 'glycogenolysis muscle'], 'tag': ['adrenal', 'pharm']},
 '3.1':  {'nome': 'T4 pro-hormonio, deiodinases, rT3', 'q': ['deiodinase', 'reverse t3', 'thyroxine', 'triiodothyronine'], 'tag': ['thyroid']},
 '3.2':  {'nome': 'Eixo TRH-TSH', 'q': ['trh', 'tsh', 'thyroid axis'], 'tag': ['thyroid']},
 '3.3':  {'nome': 'Esteroidogenese adrenal', 'q': ['21-hydroxylase', '17-hydroxylase', '11-beta-hydroxylase', 'pregnenolone', 'desmolase'], 'tag': ['adrenal']},
 '3.4':  {'nome': 'Receptor nuclear e HSP90', 'q': ['hsp90', 'heat shock protein 90', 'response element', 'coactivator'], 'tag': ['endocrine', 'biochem']},
 '3.5':  {'nome': 'Eixo HPA, ACTH/MSH, Addison', 'q': ['acth', 'addison', 'hyperpigmentation', 'pomc', 'melanocyte stimulating'], 'tag': ['adrenal']},
 '3.6':  {'nome': 'Efeitos do cortisol', 'q': ['cortisol', 'glucocorticoid', 'proteolysis muscle'], 'tag': ['adrenal']},
 '3.7':  {'nome': 'Disruptores endocrinos', 'q': ['endocrine disruptor', 'xenoestrogen', 'receptor antagonist hormone'], 'tag': ['endocrine']},
 '3.8':  {'nome': 'Efeitos do T3 (opcional)', 'q': ['thyroid hormone increases', 'thyroid hormone effect'], 'tag': ['thyroid']},
}
print(len(SPEC), 'conceitos no spec')

In [ ]:
TOP = 12          # candidatos guardados por conceito
TRECHO = 320      # caracteres do trecho enviado de volta

def pontua(nota, spec):
    t, g = nota['lower'], nota['tags_lower']
    termos = [q for q in spec['q'] if q in t]
    pistas = [p for p in spec['tag'] if p in g]
    if not termos:
        return 0, termos, pistas
    return len(termos) * 3 + len(pistas) * 2, termos, pistas

relatorio = {'slug': SLUG, 'notas_no_mestre': len(notas), 'conceitos': {}}

for cid, spec in SPEC.items():
    achados = []
    for nota in notas:
        p, termos, pistas = pontua(nota, spec)
        if p > 0:
            achados.append((p, termos, pistas, nota))
    achados.sort(key=lambda x: -x[0])
    relatorio['conceitos'][cid] = {
        'nome': spec['nome'],
        'total_encontrado': len(achados),
        'candidatos': [{
            'guid': n['guid'],
            'nid': n['nid'],
            'modelo': n['modelo'],
            'cards': n['cards'],
            'score': p,
            'termos': termos,
            'tags': n['tags'][:240],
            'trecho': n['texto'][:TRECHO],
        } for p, termos, pistas, n in achados[:TOP]],
    }

vazios = [c for c, d in relatorio['conceitos'].items() if d['total_encontrado'] == 0]
relatorio['conceitos_sem_candidato'] = vazios

destino = os.path.join(PASTA, 'candidatos-' + SLUG + '.json')
with open(destino, 'w', encoding='utf-8') as fh:
    json.dump(relatorio, fh, ensure_ascii=False, indent=1)

print('gravado:', destino)
print('tamanho:', round(os.path.getsize(destino) / 1024, 1), 'KiB')
print()
for cid, d in relatorio['conceitos'].items():
    marca = '  SEM CANDIDATO' if d['total_encontrado'] == 0 else ''
    print(f"{cid:5s} {d['total_encontrado']:5d}  {d['nome']}{marca}")
if vazios:
    print()
    print('conceitos sem candidato:', vazios)

## Depois de rodar

Me mande o `candidatos-bioq-24-controle-hormonal.json` (é pequeno, alguns KB).

Com ele eu faço a curadoria contra o contrato — escolho o card certo por conceito,
registro a rejeição real dos que não servem, decido o que sobra para autoral e
devolvo a lista de GUIDs. Aí entra o seu extrator seletivo, que já funciona, e
sai o `.apkg`.

Se algum conceito voltar com **SEM CANDIDATO**, não invente card: é sinal de que
os termos da busca erraram o vocabulário do AnKing, e o certo é ajustar o `SPEC`
daquele conceito e rodar de novo — sai bem mais barato que autorar.